In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
import lightning as L
import torch

from utils.const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import get_data, setup_device

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

loader_train = data_module.train_dataloader()
loader_val = data_module.val_dataloader()
loader_test = data_module.test_dataloader()

data_train = get_data(loader_train)
data_val = get_data(loader_val)
data_test = get_data(loader_test)

data = torch.cat([data_train, data_val, data_test], dim=0)
print(data.shape)

num_samples = data.shape[0]

What's the class distribution like?

In [ ]:
data_bg = data[:, 0, :, :].unsqueeze(1)
data_rv = data[:, 1, :, :].unsqueeze(1)
data_myo = data[:, 2, :, :].unsqueeze(1)
data_lv = data[:, 3, :, :].unsqueeze(1)

# Compute percentage of pixels that are 1

print(f"Background: {torch.mean(data_bg.float())}")
print(f"RV: {torch.mean(data_rv.float())}")
print(f"MYO: {torch.mean(data_myo.float())}")
print(f"LV: {torch.mean(data_lv.float())}")

Let's take a look at where the classes are distributed using a heatmap.

In [ ]:
heatmap_rv = torch.sum(data_rv, dim=0).squeeze() / num_samples
heatmap_myo = torch.sum(data_myo, dim=0).squeeze() / num_samples
heatmap_lv = torch.sum(data_lv, dim=0).squeeze() / num_samples

heatmap_agg = heatmap_rv + heatmap_myo + heatmap_lv

print(heatmap_rv.shape)

In [ ]:
from matplotlib import pyplot as plt

plt.style.use("ggplot")

for heatmap in [heatmap_agg, heatmap_rv, heatmap_myo, heatmap_lv]:
    plt.axis("off")
    plt.imshow(heatmap, cmap="hot", vmin=0, vmax=1)
    
    cbar = plt.colorbar()
    cbar.set_ticks([0, 0.25, 0.5, 0.75, 1])
    
    plt.tight_layout()
    plt.show()
